## Load Chunks and Model

In [1]:
import json
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATA = Path("../data")

with open(DATA / "chunks.jsonl", encoding="utf-8") as f:
	chunks = [json.loads(line) for line in f]

model = SentenceTransformer(MODEL_NAME, device="mps")   # runs on GPU
print(f"{len(chunks)} chunks, model limit: {model.max_seq_length} tokens")

W1121 11:01:34.787000 69164 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

14921 chunks, model limit: 256 tokens


## Check Truncation

In [2]:
tokenizer = model.tokenizer
# Converts a chunk into the token IDs the model actually sees; its length is the token count
lengths = np.array([len(tokenizer(c["text"])["input_ids"]) for c in chunks])
types = np.array([c["type"] for c in chunks])

for t in ["text", "table"]:
	l = lengths[types == t]
	# The share of chunks over the limit, since True counts as 1
	over = (l > model.max_seq_length).mean()
	print(f"{t:<6} median {int(np.median(l))} tokens, max {l.max()}, over limit {over:.0%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (277 > 256). Running this sequence through the model will result in indexing errors


text   median 187 tokens, max 382, over limit 1%
table  median 224 tokens, max 605, over limit 36%


## Embed All Chunks

In [3]:
start = time.time()
embeddings = model.encode(
	[c["text"] for c in chunks],
	batch_size=64,   # processes 64 chunks at a time, like the MNIST batches
	normalize_embeddings=True,   # scales every vector to length 1, so cosine similarity becomes a simple dot product
	show_progress_bar=True,
)
print(embeddings.shape, f"{time.time() - start:.0f}s")

np.save(DATA / "embeddings.npy", embeddings)

Batches:   0%|          | 0/234 [00:00<?, ?it/s]

(14921, 384) 122s


## Search Function

In [4]:
tickers = np.array([c["ticker"] for c in chunks])


def search(query, ticker, k=5):
	"""Return the top-k chunks for one company, with similarity scores."""
	q = model.encode(query, normalize_embeddings=True)
	idx = np.where(tickers == ticker)[0]   # positions of this company's chunks
	scores = embeddings[idx] @ q   # cosine similarity (vectors are normalized)
	top = np.argsort(-scores)[:k]   # best k, highest first
	return [(chunks[idx[i]], float(scores[i])) for i in top]


def show(query, ticker, k=5):
	print(f"Q: {query} [{ticker}]")
	for c, s in search(query, ticker, k):
		print(f"  {s:.3f}  {c['id']:<12} {c['type']:<5}  {c['text'][:100]!r}")
	print()

## Try Some Questions

In [5]:
show("What were Apple's total net sales in 2026?", "AAPL")             # 478,834 (AAPL-28 income statement)
show("What was diluted earnings per share in 2025?", "AAPL")           # 7.46 (AAPL-28-1)
show("How much did Apple spend on research and development in 2026?", "AAPL")  # 45,765
show("What was the gross margin percentage for Services in 2026?", "AAPL")      # 76.2%
show("What were net sales in Greater China in 2026?", "AAPL")          # 82,378
show("What are the risks related to supply chain disruptions?", "AAPL")  # text question

Q: What were Apple's total net sales in 2026? [AAPL]
  0.717  AAPL-16-0    text   '(1)Services net sales include amortization of the deferred value of services bundled in the sales pr'
  0.664  AAPL-14-1    text   'net sales of iPhone. The weakness in the yen relative to the U.S. dollar had an unfavorable year-ove'
  0.655  AAPL-12-1    text   'software offerings at various times during the year. Significant announcements during fiscal year 20'
  0.647  AAPL-12-0    text   'Item 6. [Reserved] Item 7. Management’s Discussion and Analysis of Financial Condition and Results o'
  0.622  AAPL-15-0    table  'Products and Services Performance The following table shows net sales by category for 2026, 2025 and'

Q: What was diluted earnings per share in 2025? [AAPL]
  0.741  AAPL-39-1    text   'in less than a year, 22% within one-to-two years, 11% within two-to-three years and 2% in greater th'
  0.685  AAPL-28-1    table  'CONSOLIDATED STATEMENTS OF OPERATIONS (In millions, except number of 